In [ ]:
# Run in a fresh runtime. Upload your PDF via the left Files pane or change pdf_path.

# 1) Install dependencies (correct order)
!pip install --quiet --upgrade pip
!pip install --quiet --upgrade huggingface_hub
!pip install --quiet torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
!pip install --quiet --upgrade sentence-transformers
!pip install --quiet faiss-cpu rank_bm25 pymupdf PyPDF2 transformers

# -------------------------
# Imports & helpers
# -------------------------
import os
from typing import List, Dict
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

try:
    import fitz
    PDF_ENGINE = "pymupdf"
except Exception:
    from PyPDF2 import PdfReader
    PDF_ENGINE = "pypdf2"

print("Using PDF extraction engine:", PDF_ENGINE)

def extract_text_by_page(pdf_path: str) -> List[str]:
    pages = []
    if PDF_ENGINE == "pymupdf":
        doc = fitz.open(pdf_path)
        for i, page in enumerate(doc):
            text = page.get_text("text")
            pages.append(text.strip() if text else "")
        doc.close()
    else:
        reader = PdfReader(pdf_path)
        for i, page in enumerate(reader.pages):
            try:
                text = page.extract_text() or ""
            except Exception:
                text = ""
            pages.append(text.strip())
    return pages

# -------------------------
# Chunking strategy
# -------------------------
def chunk_pages_to_passages(pages: List[str], max_tokens_approx: int = 300) -> List[Dict]:
    import re
    passages = []
    pid = 0
    for pno, page in enumerate(pages, start=1):
        text = page.replace("\r", "\n")
        sentences = [s.strip() for s in re.split(r'(?<=[\.\?\!])\s+', text) if s.strip()]
        if not sentences:
            continue
        # If page short, keep as single passage
        if len(text.split()) <= max_tokens_approx:
            passages.append({"id": pid, "page": pno, "text": text})
            pid += 1
            continue
        # otherwise sliding window of sentences
        window = []
        wcount = 0
        for s in sentences:
            s_len = len(s.split())
            if wcount + s_len <= max_tokens_approx:
                window.append(s)
                wcount += s_len
            else:
                passages.append({"id": pid, "page": pno, "text": " ".join(window)})
                pid += 1
                window = [s]
                wcount = s_len
        if window:
            passages.append({"id": pid, "page": pno, "text": " ".join(window)})
            pid += 1
    return passages

# -------------------------
# Embedding model + index build
# -------------------------
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
print("Loading embedding model:", EMBEDDING_MODEL_NAME)
embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)

def build_faiss_index(passages: List[Dict], embedder_model: SentenceTransformer):
    texts = [p["text"] for p in passages]
    embeddings = embedder_model.encode(texts, convert_to_numpy=True, show_progress_bar=True, batch_size=32)
    d = embeddings.shape[1]
    index = faiss.IndexFlatIP(d)  # inner product on normalized vectors -> cosine
    faiss.normalize_L2(embeddings)
    index.add(embeddings)
    return index, embeddings

# -------------------------
# BM25 index build
# -------------------------
def build_bm25(passages: List[Dict]):
    tokenized = [p["text"].lower().split() for p in passages]
    bm25 = BM25Okapi(tokenized)
    return bm25, tokenized

# -------------------------
# Hybrid retrieval: vector (FAISS) + BM25
# -------------------------
def hybrid_retrieve(query: str,
                    passages: List[Dict],
                    index,
                    embeddings,
                    embedder_model: SentenceTransformer,
                    bm25=None,
                    tokenized_bm25=None,
                    top_k: int = 6) -> List[Dict]:
    q_emb = embedder_model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)
    D, I = index.search(q_emb, top_k)
    vector_idxs = I[0].tolist()
    vector_candidates = [passages[i] for i in vector_idxs if i < len(passages)]

    bm25_candidates = []
    if bm25 is not None:
        tokenized_query = query.lower().split()
        bm25_scores = bm25.get_scores(tokenized_query)
        top_bm25_idx = np.argsort(bm25_scores)[::-1][:top_k]
        bm25_candidates = [passages[int(i)] for i in top_bm25_idx if int(i) < len(passages)]

    combined = []
    seen = set()
    for p in vector_candidates + bm25_candidates:
        key = (p["page"], p["text"][:200])
        if key not in seen:
            combined.append(p)
            seen.add(key)
    return combined

# -------------------------
# Reranking using Cross Encoder
# -------------------------
RERANKER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
print("Loading reranker:", RERANKER_NAME)
reranker = CrossEncoder(RERANKER_NAME)

def rerank_candidates(query: str, candidates: List[Dict], reranker_model: CrossEncoder, top_n: int = 3):
    if not candidates:
        return []
    texts = [c["text"] for c in candidates]
    pairs = [[query, t] for t in texts]
    scores = reranker_model.predict(pairs)  # higher is better
    idxs = np.argsort(scores)[::-1][:top_n]
    reranked = [candidates[int(i)] for i in idxs]
    for pos, i in enumerate(idxs):
        reranked[pos]["score"] = float(scores[int(i)])
    return reranked

# -------------------------
# Full pipeline runner
# -------------------------
def build_and_run_pipeline(pdf_path: str,
                           max_tokens_page: int = 300,
                           top_k_retrieval: int = 6,
                           rerank_top_n: int = 3):
    assert os.path.exists(pdf_path), f"PDF not found at {pdf_path}"
    print("Extracting PDF pages...")
    pages = extract_text_by_page(pdf_path)
    print(f"Extracted {len(pages)} pages.")
    print("Chunking pages into passages...")
    passages = chunk_pages_to_passages(pages, max_tokens_approx=max_tokens_page)
    print(f"Created {len(passages)} passages / chunks.")
    if len(passages) == 0:
        raise RuntimeError("No text extracted from PDF.")

    print("Building FAISS vector index (this will compute embeddings)...")
    index, embeddings = build_faiss_index(passages, embedder)

    print("Building BM25 index...")
    bm25, tokenized = build_bm25(passages)

    pipeline = {
        "passages": passages,
        "faiss_index": index,
        "embeddings": embeddings,
        "bm25": bm25,
        "tokenized": tokenized,
        "embedder": embedder,
        "reranker": reranker
    }
    return pipeline

def answer_query(pipeline: dict, query: str, top_k_retrieval: int = 6, rerank_top_n: int = 3):
    passages = pipeline["passages"]
    idx = pipeline["faiss_index"]
    emb = pipeline["embeddings"]
    bm25 = pipeline["bm25"]
    embedder_model = pipeline["embedder"]
    reranker_model = pipeline["reranker"]

    candidates = hybrid_retrieve(query, passages, idx, emb, embedder_model, bm25=bm25, tokenized_bm25=pipeline["tokenized"], top_k=top_k_retrieval)
    print(f"Hybrid retrieval returned {len(candidates)} candidates.")
    reranked = rerank_candidates(query, candidates, reranker_model, top_n=rerank_top_n)
    print(f"Reranked top {len(reranked)} passages.")
    for i, p in enumerate(reranked, start=1):
        print("="*80)
        score = p.get('score', None)
        score_str = f"{score:.4f}" if score is not None else "N/A"
        print(f"Rank {i} — page {p.get('page')} — score: {score_str}")
        print(p["text"][:2000])
        print("\n")
    return reranked

# -------------------------
# Run example (build index then answer two target prompts)
# -------------------------
pdf_path = "sample-sdf-document (Project 5).pdf"  # update if different
if os.path.exists(pdf_path):
    print("Building pipeline from:", pdf_path)
    pipeline = build_and_run_pipeline(pdf_path)
    prompt1 = "What test methods were used for quality control?"
    prompt2 = "What are the storage conditions specified in the certificate?"
    print("\n\n=== Prompt 1 ===")
    answer_query(pipeline, prompt1, top_k_retrieval=8, rerank_top_n=4)
    print("\n\n=== Prompt 2 ===")
    answer_query(pipeline, prompt2, top_k_retrieval=8, rerank_top_n=4)
else:
    print(f"PDF file not found at '{pdf_path}'. Please upload it via the Colab left pane (Files) or change pdf_path variable.")